# Lab 14: Multimodal LLMs for Visual Question Answering

In [ ]:
!pip install -q transformers accelerate pillow requests matplotlib

In [ ]:
from io import BytesIO

import matplotlib.pyplot as plt
import requests
from PIL import Image, ImageDraw, ImageFont
from transformers import pipeline

pipe = pipeline(
    "image-text-to-text",
    model="HuggingFaceTB/SmolVLM-256M-Instruct",
    device_map="auto"
)


def load_image_from_url(url: str) -> Image.Image:
    response = requests.get(url, timeout=20)
    response.raise_for_status()
    return Image.open(BytesIO(response.content)).convert("RGB")


def ask_image(image: Image.Image, question: str, max_new_tokens: int = 100) -> str:
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": question},
            ],
        }
    ]
    output = pipe(
        text=messages,
        images=[image],
        max_new_tokens=max_new_tokens,
        return_full_text=False,
    )
    return output[0]["generated_text"]


def combine_side_by_side(left: Image.Image, right: Image.Image) -> Image.Image:
    canvas = Image.new("RGB", (left.width + right.width, max(left.height, right.height)), "white")
    canvas.paste(left, (0, 0))
    canvas.paste(right, (left.width, 0))
    return canvas


print("Multimodal pipeline ready.")

## Task 1: Basic Image Description

In [ ]:
image_url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/bee.jpg"
image = load_image_from_url(image_url)

display(image)

description = ask_image(image, "Describe this image in detail.")
print("Image Description:")
print(description)

## Task 2: Visual Question Answering on a Chart

In [ ]:
months = ["Jan", "Feb", "Mar", "Apr", "May"]
sales = [12, 18, 15, 22, 30]

plt.figure(figsize=(7, 4))
plt.plot(months, sales, marker="o", linewidth=2)
plt.title("Monthly Sales")
plt.xlabel("Month")
plt.ylabel("Sales")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("sales_chart.png")
plt.show()

chart_image = Image.open("sales_chart.png").convert("RGB")
display(chart_image)

questions = [
    "What does this chart show?",
    "Which month appears to have the highest sales?",
    "Describe the trend shown in the chart.",
]

for question in questions:
    print("Q:", question)
    print("A:", ask_image(chart_image, question))
    print("-" * 60)

## Task 3: OCR Style Text Reading

In [ ]:
img = Image.new("RGB", (500, 220), color="white")
draw = ImageDraw.Draw(img)
font = ImageFont.load_default()

draw.text((20, 40), "CS3240 Lab 14", fill="black", font=font)
draw.text((20, 90), "Topic: Visual Question Answering", fill="black", font=font)
draw.text((20, 140), "Student Demo Image", fill="black", font=font)

img.save("ocr_demo.png")
display(img)

ocr_result = ask_image(img, "Extract all visible text from this image.")
print("OCR Result:")
print(ocr_result)

## Task 4: Visual Reasoning by Comparing Two Images

In [ ]:
img1 = Image.new("RGB", (260, 220), color="white")
draw1 = ImageDraw.Draw(img1)
draw1.rectangle((30, 40, 110, 120), outline="blue", width=4)
draw1.ellipse((150, 60, 220, 130), outline="green", width=4)
draw1.text((70, 170), "Left", fill="black", font=ImageFont.load_default())

img2 = Image.new("RGB", (260, 220), color="white")
draw2 = ImageDraw.Draw(img2)
draw2.rectangle((40, 50, 130, 140), outline="red", width=4)
draw2.ellipse((160, 40, 230, 110), outline="purple", width=4)
draw2.text((70, 170), "Right", fill="black", font=ImageFont.load_default())

comparison_image = combine_side_by_side(img1, img2)
comparison_image.save("comparison.png")
display(comparison_image)

comparison_answer = ask_image(
    comparison_image,
    "Compare the left and right halves of this image. What are the main similarities and differences?"
)
print("Comparison Answer:")
print(comparison_answer)

## Task 5: Analyze a Local Image File

In [ ]:
local_image_path = "ocr_demo.png"
local_image = Image.open(local_image_path).convert("RGB")
display(local_image)

local_answer = ask_image(
    local_image,
    "What does this local image contain? Mention any text you can read."
)
print("Local Image Analysis:")
print(local_answer)

## Task 6: Image Based Code Generation

In [ ]:
diagram = Image.new("RGB", (520, 260), color="lightyellow")
draw = ImageDraw.Draw(diagram)
font = ImageFont.load_default()

draw.rectangle((40, 40, 180, 90), outline="black", width=3)
draw.text((75, 58), "Start", fill="black", font=font)

draw.rectangle((190, 40, 340, 90), outline="black", width=3)
draw.text((210, 58), "Read number n", fill="black", font=font)

draw.rectangle((350, 40, 500, 90), outline="black", width=3)
draw.text((365, 58), "Double n", fill="black", font=font)

draw.rectangle((190, 150, 340, 210), outline="black", width=3)
draw.text((215, 175), "Print result", fill="black", font=font)

draw.line((180, 65, 190, 65), fill="black", width=3)
draw.line((340, 65, 350, 65), fill="black", width=3)
draw.line((425, 90, 425, 130), fill="black", width=3)
draw.line((425, 130, 265, 130), fill="black", width=3)
draw.line((265, 130, 265, 150), fill="black", width=3)

diagram.save("diagram.png")
display(diagram)

code_from_diagram = ask_image(
    diagram,
    "Generate Python code that matches the flow shown in this diagram."
)
print("Generated Code:")
print(code_from_diagram)